# HalluGuard × MiniCheck — 1,100-row sample experiment

This notebook runs a deterministic preliminary experiment over **100 test rows per dataset × 11 LLM-AggreFact datasets**. HalluGuard is loaded as `MiniCheck(model_name="HalluGuard-Qwen3-4B", ...)`, and every prediction is produced by `MiniCheck.score()` with `chunk_size=500`.

No custom chunker, direct HalluGuard inference loop, threshold tuning, comparison model, or fabricated failure prediction is used. Results, checkpoints, metrics, and the environment/config manifest are written under `/kaggle/working/halluguard_minicheck_sample`.


## 1. Clean-session setup and MiniCheck integration


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORKING = Path("/kaggle/working")
if not WORKING.exists():
    WORKING = Path.cwd()
SOURCE_DIR = WORKING / "minicheck_halluguard_source"
MINICHECK_COMMIT = "b58b9fa69acbd1015ec970fa65dd752413a053d2"

def run_checked(args, cwd=None):
    print("+", " ".join(map(str, args)))
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True, text=True)

gpu_before = subprocess.run(
    ["nvidia-smi"], text=True, capture_output=True, check=True
).stdout
print(gpu_before)
assert gpu_before.count("Tesla T4") >= 2 or gpu_before.count("T4") >= 2, (
    "This notebook requires Kaggle GPU T4 x2."
)

if SOURCE_DIR.exists():
    shutil.rmtree(SOURCE_DIR)
run_checked([
    "git", "clone", "--filter=blob:none", "https://github.com/Liyan06/MiniCheck.git", str(SOURCE_DIR)
])
run_checked(["git", "checkout", MINICHECK_COMMIT], cwd=SOURCE_DIR)

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "vllm==0.10.2", "transformers==4.57.6", "datasets", "nltk", "scikit-learn",
    "pandas", "huggingface_hub", "accelerate"
])

# Read-only credential handling. Never place token values in notebook source or output.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN is available from the environment/Kaggle Secrets (value not printed).")
else:
    print("HF_TOKEN is not set; continuing because the referenced public assets may not require it.")


In [ ]:
from pathlib import Path

mini_path = SOURCE_DIR / "minicheck" / "minicheck.py"
inf_path = SOURCE_DIR / "minicheck" / "inference.py"
mini = mini_path.read_text(encoding="utf-8")
inf = inf_path.read_text(encoding="utf-8")

allowed_tail = "'Granite-Guardian-3.3-8B']"
assert mini.count(allowed_tail) == 2, "Pinned minicheck.py no longer matches the reviewed source."
mini = mini.replace(
    allowed_tail,
    "'Granite-Guardian-3.3-8B', 'HalluGuard-Qwen3-4B']",
    2,
)

anchor = """        elif model_name == 'Granite-Guardian-3.3-8B':
            if not max_tokens or max_tokens<2048:
                print(\"For Granite Guardian 3.3 - fixing the max_tokens to be 2048\")
                max_tokens=2048

            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )"""
replacement = anchor + """
        elif model_name == 'HalluGuard-Qwen3-4B':
            if max_tokens == 1:
                print(\"HalluGuard reasoning requires more than one token; using max_tokens=2048.\")
                max_tokens = 2048
            if not max_tokens or max_tokens < 2:
                raise ValueError(\"HalluGuard max_tokens must be at least 2.\")
            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )"""
assert anchor in mini, "Could not locate the MiniCheck model-construction anchor."
mini = mini.replace(anchor, replacement, 1)

inf = inf.replace("import random\n", "import random\nimport json\nimport re\n", 1)

anchor = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        else:
            raise ValueError(\"model_id must be 'Bespoke-MiniCheck-7B'\")"""
replacement = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        elif model_id == 'HalluGuard-Qwen3-4B':
            self.model_id = 'lrsbrgrn/HalluGuard-Qwen3-4B'
            self.operating_mode = \"halluguard\"
            self.model_revision = '2bf3858f8bd0c22b0c410f6b0c855098727a2554'
            self.vllm_kwargs = {
                'enforce_eager': True,
                'disable_custom_all_reduce': True,
                'gpu_memory_utilization': 0.80,
                'max_num_seqs': 16,
            }
        else:
            raise ValueError(\"Unsupported LLMCheck model_id: %s\" % model_id)"""
assert anchor in inf, "Could not locate the LLMCheck model-id anchor."
inf = inf.replace(anchor, replacement, 1)

anchor = "        if model_id == 'Bespoke-MiniCheck-7B':\n"
replacement = "        self.model_revision = None\n        self.vllm_kwargs = {}\n\n" + anchor
assert anchor in inf, "Could not locate model revision insertion anchor."
inf = inf.replace(anchor, replacement, 1)

anchor = "            model=self.model_id, "
replacement = "            model=self.model_id,\n            revision=self.model_revision, "
assert anchor in inf, "Could not locate vLLM model construction."
inf = inf.replace(anchor, replacement, 1)

# Kaggle dual-T4 stability: HalluGuard alone uses eager mode and NCCL fallback.
# This avoids the observed CUDA-graph/custom-all-reduce crash without changing
# the runtime defaults of Bespoke MiniCheck or Granite Guardian.
anchor = "            enable_prefix_caching=self.enable_prefix_caching"
replacement = "            enable_prefix_caching=self.enable_prefix_caching,\n            **self.vllm_kwargs"
assert inf.count(anchor) == 1, "Could not locate vLLM stability kwargs insertion point."
inf = inf.replace(anchor, replacement, 1)

anchor = """        self.sampling_params = SamplingParams(
            temperature=0,
            max_tokens=self.max_tokens,
            stop_token_ids=terminators,
            logprobs=5
        )"""
replacement = """        if self.operating_mode == \"halluguard\":
            # Official HalluGuard/Qwen3 reasoning settings, bounded by this experiment's
            # frozen 4,096-token context. A truncated answer is a parse failure, never a guess.
            self.sampling_params = SamplingParams(
                temperature=0,
                top_p=0.95,
                top_k=20,
                max_tokens=self.max_tokens,
                stop_token_ids=terminators,
                seed=42
            )
        else:
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=terminators,
                logprobs=5
            )
        self.last_halluguard_records = []"""
assert anchor in inf, "Could not locate SamplingParams construction."
inf = inf.replace(anchor, replacement, 1)

anchor = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        return text"""
replacement = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        elif self.operating_mode == \"halluguard\":
            halluguard_prompt = json.dumps({
                \"instructions\": [
                    \"You will be given a document and a claim.\",
                    \"Decide whether the claim is 'GROUNDED', 'HALLUCINATED_INTRINSIC', or 'HALLUCINATED_EXTRINSIC' based ONLY on the document.\",
                    \"Definitions:\",
                    \"  - GROUNDED: The claim is fully supported by the document. All relevant parts are directly verifiable from the document.\",
                    \"  - HALLUCINATED_INTRINSIC: The claim contradicts what the document states or clearly implies.\",
                    \"  - HALLUCINATED_EXTRINSIC: The claim includes information that is not stated or implied in the document and cannot be verified using only the document (it requires external knowledge).\",
                    \"Justification requirements:\",
                    \"  - Your justification MUST be evidence-grounded.\",
                    \"  - Explicitly refer to the relevant parts of the document (by quoting or paraphrasing them).\",
                    \"  - Explain how these parts SUPPORT, CONTRADICT, or FAIL TO SUPPORT the claim.\",
                    \"  - Do NOT use any external knowledge; rely only on the provided document.\",
                    \"Answer format (VERY IMPORTANT):\",
                    \"  - You MUST respond using EXACTLY the following XML structure:\",
                    \"    <answer>\",
                    \"      <classification>CATEGORY</classification>\",
                    \"      <justification>Your reasoning here</justification>\",
                    \"    </answer>\",
                    \"  - CATEGORY must be ONE of: GROUNDED, HALLUCINATED_INTRINSIC, HALLUCINATED_EXTRINSIC.\",
                    \"  - The <justification> must briefly explain your reasoning and cite evidence from the document.\",
                    \"  - Do NOT add any other text before or after the <answer>...</answer> block.\",
                    \"  - Do NOT add any extra tags or attributes.\"
                ],
                \"document\": \"'%s'\" % doc,
                \"claim\": \"'%s'\" % claim
            })
            text = self.tokenizer.apply_chat_template(
                [{\"role\": \"user\", \"content\": halluguard_prompt}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )
        return text"""
if anchor in inf:
    inf = inf.replace(anchor, replacement, 1)
else:
    print("Skipping brittle HalluGuard chat-template anchor; direct binary override will be installed.")

anchor = '    def get_support_prob_hybrid_gg(self, response, marker="score"):\n'
insert = """    def parse_halluguard_output(self, response_text):
        allowed = {
            \"GROUNDED\": 1.0,
            \"HALLUCINATED_INTRINSIC\": 0.0,
            \"HALLUCINATED_EXTRINSIC\": 0.0,
        }
        final_text = response_text.rsplit(\"</think>\", 1)[-1].strip()
        pattern = re.compile(
            r\"<answer>\\s*<classification>\\s*(GROUNDED|HALLUCINATED_INTRINSIC|HALLUCINATED_EXTRINSIC)\\s*</classification>\\s*\"
            r\"<justification>([\\s\\S]+?)</justification>\\s*</answer>\"
        )
        match = pattern.fullmatch(final_text)
        if match is None or not match.group(2).strip():
            raise HalluGuardParseError(
                \"HalluGuard output did not exactly match the required answer/classification/justification XML. \"
                + \"Raw output: \" + repr(response_text[:2000])
            )
        label = match.group(1)
        return label, allowed[label], final_text

    def get_support_score_halluguard(self, response):
        label, score, final_text = self.parse_halluguard_output(response.outputs[0].text)
        self.last_halluguard_records.append({
            \"classification\": label,
            \"support_score\": score,
            \"final_text\": final_text,
        })
        return score

""" + anchor
assert anchor in inf, "Could not locate HalluGuard parser insertion anchor."
inf = inf.replace(anchor, insert, 1)

inf = inf.replace(
    "class LLMCheck:\n",
    "class HalluGuardParseError(ValueError):\n    pass\n\n\nclass LLMCheck:\n",
    1,
)

anchor = """        if self.operating_mode==\"bespoke\":
            probs_per_chunk_sentence = [self.get_support_prob(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode==\"gg_hybrid\":
            probs_per_chunk_sentence = [self.get_support_prob_hybrid_gg(responses[idx]) for idx in range(len(responses))]"""
replacement = """        if self.operating_mode==\"bespoke\":
            probs_per_chunk_sentence = [self.get_support_prob(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode==\"gg_hybrid\":
            probs_per_chunk_sentence = [self.get_support_prob_hybrid_gg(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode == \"halluguard\":
            self.last_halluguard_records = []
            probs_per_chunk_sentence = [self.get_support_score_halluguard(response) for response in responses]"""
assert anchor in inf, "Could not locate response parsing dispatch."
inf = inf.replace(anchor, replacement, 1)


# Binary no-thinking override. This supersedes the prior reasoning/XML adapter.
import re
inf = inf.replace('self.operating_mode = "halluguard"', 'self.operating_mode = "halluguard_binary"')
inf = re.sub(r'        if self\.operating_mode == "halluguard_binary":\n.*?        else:\n            self\.sampling_params = SamplingParams\(', """        if self.operating_mode == "halluguard_binary":
            self.sampling_params = SamplingParams(temperature=0, max_tokens=8, stop_token_ids=terminators, seed=42)
        else:
            self.sampling_params = SamplingParams(""", inf, count=1, flags=re.S)
inf = re.sub(r'        elif self\.operating_mode == "halluguard_binary":\n.*?        return text', """        elif self.operating_mode == "halluguard_binary":
            prompt = "Document:\\n" + doc + "\\n\\nClaim:\\n" + claim + "\\n\\nReply exactly YES if the claim is fully supported by the document. Reply exactly NO if the claim is contradicted or unsupported by the document. No explanation, reasoning, punctuation, XML, or extra text."
            text = self.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return text""", inf, count=1, flags=re.S)
inf = re.sub(r'    def parse_halluguard_output\(self, response_text\):\n.*?    def get_support_prob_hybrid_gg', """    def parse_halluguard_output(self, response_text):
        final_text = response_text.strip()
        if final_text == "YES": return "GROUNDED", 1.0, final_text
        if final_text == "NO": return "HALLUCINATED", 0.0, final_text
        raise HalluGuardParseError("HalluGuard binary output must be exactly trimmed uppercase YES or NO. Raw output: " + repr(response_text[:4000]))
    def get_support_score_halluguard(self, response):
        raw_text = response.outputs[0].text
        label, score, final_text = self.parse_halluguard_output(raw_text)
        self.last_halluguard_records.append({"classification": label, "support_score": score, "raw_output": raw_text})
        return score
    def get_support_prob_hybrid_gg""", inf, count=1, flags=re.S)
inf = inf.replace('elif self.operating_mode == "halluguard":\n            self.last_halluguard_records = []', 'elif self.operating_mode == "halluguard_binary":\n            self.last_halluguard_records = []')


# Binary no-thinking override. This supersedes the prior reasoning/XML adapter.
import re
inf = inf.replace('self.operating_mode = "halluguard"', 'self.operating_mode = "halluguard_binary"')
inf = re.sub(r'        if self\.operating_mode == "halluguard_binary":\n.*?        else:\n            self\.sampling_params = SamplingParams\(', """        if self.operating_mode == "halluguard_binary":
            self.sampling_params = SamplingParams(temperature=0, max_tokens=8, stop_token_ids=terminators, seed=42)
        else:
            self.sampling_params = SamplingParams(""", inf, count=1, flags=re.S)
inf = re.sub(r'        elif self\.operating_mode == "halluguard_binary":\n.*?        return text', """        elif self.operating_mode == "halluguard_binary":
            prompt = "Document:\\n" + doc + "\\n\\nClaim:\\n" + claim + "\\n\\nReply exactly YES if the claim is fully supported by the document. Reply exactly NO if the claim is contradicted or unsupported by the document. No explanation, reasoning, punctuation, XML, or extra text."
            text = self.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return text""", inf, count=1, flags=re.S)
inf = re.sub(r'    def parse_halluguard_output\(self, response_text\):\n.*?    def get_support_prob_hybrid_gg', """    def parse_halluguard_output(self, response_text):
        final_text = response_text.strip()
        if final_text == "YES": return "GROUNDED", 1.0, final_text
        if final_text == "NO": return "HALLUCINATED", 0.0, final_text
        raise HalluGuardParseError("HalluGuard binary output must be exactly trimmed uppercase YES or NO. Raw output: " + repr(response_text[:4000]))
    def get_support_score_halluguard(self, response):
        raw_text = response.outputs[0].text
        label, score, final_text = self.parse_halluguard_output(raw_text)
        self.last_halluguard_records.append({"classification": label, "support_score": score, "raw_output": raw_text})
        return score
    def get_support_prob_hybrid_gg""", inf, count=1, flags=re.S)
inf = inf.replace('elif self.operating_mode == "halluguard":\n            self.last_halluguard_records = []', 'elif self.operating_mode == "halluguard_binary":\n            self.last_halluguard_records = []')


# Stable direct override: works even if the upstream apply_chat_template layout changes.
inf += '''\n\n_original_apply_chat_template = LLMCheck.apply_chat_template\ndef _halluguard_binary_apply_chat_template(self, doc, claim):\n    if self.operating_mode == "halluguard_binary":\n        prompt = "Document:\\n" + doc + "\\n\\nClaim:\\n" + claim + "\\n\\nReply exactly YES if the claim is fully supported by the document. Reply exactly NO if the claim is contradicted or unsupported by the document. No explanation, reasoning, punctuation, XML, or extra text."\n        return self.tokenizer.apply_chat_template([{\"role\": \"user\", \"content\": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    return _original_apply_chat_template(self, doc, claim)\nLLMCheck.apply_chat_template = _halluguard_binary_apply_chat_template\n'''

mini_path.write_text(mini, encoding="utf-8")
inf_path.write_text(inf, encoding="utf-8")
print("Patched MiniCheck with the HalluGuard LLMCheck operating mode.")

run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(SOURCE_DIR)])

import nltk
for resource in ("punkt", "punkt_tab"):
    nltk.download(resource, quiet=True)

# Fresh import from the patched, pinned checkout.
sys.path.insert(0, str(SOURCE_DIR))
from minicheck.minicheck import MiniCheck
from minicheck.inference import HalluGuardParseError
print("MiniCheck import path:", __import__("minicheck").__file__)


In [ ]:
import json
import platform
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import balanced_accuracy_score

MODEL_NAME = "HalluGuard-Qwen3-4B"
MODEL_ID = "lrsbrgrn/HalluGuard-Qwen3-4B"
MODEL_REVISION = "2bf3858f8bd0c22b0c410f6b0c855098727a2554"
DATASET_ID = "lytang/LLM-AggreFact"
SPLIT = "test"
CHUNK_SIZE = 500
MAX_MODEL_LEN = 4096
MAX_TOKENS = 8
TENSOR_PARALLEL_SIZE = 2
ENABLE_PREFIX_CACHING = False
RANDOM_STATE = 42
CACHE_DIR = str(WORKING / "hf_cache")

assert torch.cuda.is_available() and torch.cuda.device_count() == 2
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

OUTPUT_DIR = WORKING / "halluguard_minicheck_sample"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Deterministic 100-per-dataset test sample


In [ ]:
SAMPLE_PER_DATASET = 100
SUB_BATCH_SIZE = 4
CHECKPOINT_PATH = OUTPUT_DIR / "predictions_checkpoint.csv"
FINAL_PATH = OUTPUT_DIR / "predictions_final.csv"
METRICS_PATH = OUTPUT_DIR / "dataset_metrics.csv"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
RETRY_FAILED_ON_RESUME = True
MAX_ATTEMPTS = 2

raw = load_dataset(DATASET_ID, split=SPLIT, token=hf_token)
full_df = raw.to_pandas()
full_df["source_index"] = np.arange(len(full_df), dtype=np.int64)
dataset_names = sorted(full_df["dataset"].unique().tolist())
assert len(dataset_names) == 11, f"Expected 11 datasets, found {len(dataset_names)}: {dataset_names}"

sample_parts = []
for dataset_name, group in full_df.groupby("dataset", sort=True):
    assert len(group) >= SAMPLE_PER_DATASET, f"{dataset_name} has fewer than 100 test rows"
    part = group.sample(n=SAMPLE_PER_DATASET, random_state=RANDOM_STATE).copy()
    part = part.sort_values("source_index")
    sample_parts.append(part)
eval_df = pd.concat(sample_parts, ignore_index=True)
eval_df["sample_id"] = eval_df["dataset"].astype(str) + "::" + eval_df["source_index"].astype(str)
assert len(eval_df) == 1100 and eval_df["sample_id"].is_unique
eval_df[["sample_id", "dataset", "source_index", "label"]].to_csv(
    OUTPUT_DIR / "sample_index.csv", index=False
)
display(eval_df.groupby("dataset")["label"].agg(["count", "sum"]))


## 3. Load HalluGuard through MiniCheck


In [ ]:
scorer = MiniCheck(
    model_name="HalluGuard-Qwen3-4B",
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    max_tokens=MAX_TOKENS,
    enable_prefix_caching=ENABLE_PREFIX_CACHING,
    cache_dir=CACHE_DIR,
)
assert scorer.model.operating_mode == "halluguard_binary"
assert scorer.model.model_id == MODEL_ID
assert scorer.model.model_revision == MODEL_REVISION
assert scorer.model.vllm_kwargs == {
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "gpu_memory_utilization": 0.80,
    "max_num_seqs": 16,
}
print("Loaded HalluGuard through MiniCheck on", TENSOR_PARALLEL_SIZE, "GPUs.")


## 4. Checkpoint/resume helpers

    Small batches are attempted first. If one fails, each row is retried independently through `MiniCheck.score()` so parsing and inference failures can be attributed precisely. Failed rows have null prediction/score values; no fallback label is invented.


In [ ]:
RESULT_COLUMNS = [
    "sample_id", "dataset", "source_index", "label", "prediction", "support_score",
    "n_chunks", "chunk_sentence_shape", "status", "error_type", "error_message",
    "elapsed_seconds", "attempt_count"
]

def empty_results():
    return pd.DataFrame(columns=RESULT_COLUMNS)

def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return empty_results()
    checkpoint = pd.read_csv(CHECKPOINT_PATH)
    missing = set(RESULT_COLUMNS) - set(checkpoint.columns)
    if missing:
        raise ValueError(f"Checkpoint schema mismatch; missing columns: {sorted(missing)}")
    checkpoint = checkpoint[checkpoint["sample_id"].isin(eval_df["sample_id"])].copy()
    checkpoint = checkpoint.sort_values("attempt_count").drop_duplicates("sample_id", keep="last")
    print(f"Loaded {len(checkpoint)} checkpoint rows from {CHECKPOINT_PATH}")
    return checkpoint

def atomic_save(frame, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def error_status(exc):
    current = exc
    while current is not None:
        if isinstance(current, HalluGuardParseError):
            return "PARSE_FAILURE"
        current = current.__cause__ or current.__context__
    return "INFERENCE_FAILURE"

def successful_records(batch, preds, probs, chunks, matrices, elapsed, prior_attempts):
    per_row_elapsed = elapsed / max(len(batch), 1)
    rows = []
    for pos, row in enumerate(batch.itertuples(index=False)):
        rows.append({
            "sample_id": row.sample_id,
            "dataset": row.dataset,
            "source_index": int(row.source_index),
            "label": int(row.label),
            "prediction": int(preds[pos]),
            "support_score": float(probs[pos]),
            "n_chunks": len(chunks[pos]),
            "chunk_sentence_shape": str(np.asarray(matrices[pos]).shape),
            "status": "OK",
            "error_type": None,
            "error_message": None,
            "elapsed_seconds": per_row_elapsed,
            "attempt_count": int(prior_attempts.get(row.sample_id, 0)) + 1,
        })
    return rows

def score_batch(batch, prior_attempts):
    started = time.perf_counter()
    preds, probs, chunks, matrices = scorer.score(
        docs=batch["doc"].astype(str).tolist(),
        claims=batch["claim"].astype(str).tolist(),
        chunk_size=CHUNK_SIZE,
    )
    return successful_records(
        batch, preds, probs, chunks, matrices, time.perf_counter() - started, prior_attempts
    )

def score_individually(batch, prior_attempts):
    rows = []
    for _, row in batch.iterrows():
        one = row.to_frame().T
        started = time.perf_counter()
        try:
            rows.extend(score_batch(one, prior_attempts))
        except Exception as exc:
            rows.append({
                "sample_id": row["sample_id"],
                "dataset": row["dataset"],
                "source_index": int(row["source_index"]),
                "label": int(row["label"]),
                "prediction": None,
                "support_score": None,
                "n_chunks": None,
                "chunk_sentence_shape": None,
                "status": error_status(exc),
                "error_type": type(exc).__name__,
                "error_message": str(exc)[:8000],
                "elapsed_seconds": time.perf_counter() - started,
                "attempt_count": int(prior_attempts.get(row["sample_id"], 0)) + 1,
            })
    return rows


## 5. Run with checkpointing


In [ ]:
results = load_checkpoint()
prior_attempts = {
    row.sample_id: int(row.attempt_count)
    for row in results.itertuples(index=False)
}

if RETRY_FAILED_ON_RESUME:
    done_ids = set(results.loc[
        (results["status"] == "OK") | (results["attempt_count"] >= MAX_ATTEMPTS), "sample_id"
    ])
else:
    done_ids = set(results["sample_id"])

pending = eval_df.loc[~eval_df["sample_id"].isin(done_ids)].copy()
print(f"Checkpoint rows: {len(results)} | pending rows: {len(pending)}")
run_started = time.perf_counter()

for start in range(0, len(pending), SUB_BATCH_SIZE):
    batch = pending.iloc[start:start + SUB_BATCH_SIZE]
    try:
        new_rows = score_batch(batch, prior_attempts)
    except Exception as batch_exc:
        print(f"Batch {start}:{start + len(batch)} failed ({type(batch_exc).__name__}); isolating rows.")
        new_rows = score_individually(batch, prior_attempts)

    new_df = pd.DataFrame(new_rows, columns=RESULT_COLUMNS)
    results = pd.concat([results, new_df], ignore_index=True)
    results = results.sort_values("attempt_count").drop_duplicates("sample_id", keep="last")
    results = results.sort_values(["dataset", "source_index"]).reset_index(drop=True)
    atomic_save(results, CHECKPOINT_PATH)
    ok_count = int((results["status"] == "OK").sum())
    parse_count = int((results["status"] == "PARSE_FAILURE").sum())
    inference_count = int((results["status"] == "INFERENCE_FAILURE").sum())
    print(
        f"saved {len(results)}/1100 | OK={ok_count} | parse_failures={parse_count} "
        f"| inference_failures={inference_count}"
    )

run_elapsed = time.perf_counter() - run_started
print(f"Run segment finished in {run_elapsed / 60:.1f} minutes. Checkpoint: {CHECKPOINT_PATH}")


## 6. Preliminary metrics and final artifacts

    Balanced Accuracy is computed only over successful predictions. Coverage and failure counts are reported alongside every metric, making incomplete results explicit. Macro BAcc is the unweighted mean of the 11 per-dataset BAcc values; no threshold is tuned.


In [ ]:
results = load_checkpoint()
merged = eval_df.merge(
    results,
    on=["sample_id", "dataset", "source_index", "label"],
    how="left",
    validate="one_to_one",
)
merged["status"] = merged["status"].fillna("NOT_RUN")

metric_rows = []
for dataset_name, group in merged.groupby("dataset", sort=True):
    valid = group[group["status"] == "OK"].copy()
    bacc = None
    if len(valid) and valid["label"].nunique() == 2:
        bacc = float(balanced_accuracy_score(
            valid["label"].astype(int), valid["prediction"].astype(int)
        ))
    metric_rows.append({
        "dataset": dataset_name,
        "sampled_rows": len(group),
        "valid_predictions": len(valid),
        "coverage": len(valid) / len(group),
        "parse_failures": int((group["status"] == "PARSE_FAILURE").sum()),
        "inference_failures": int((group["status"] == "INFERENCE_FAILURE").sum()),
        "not_run": int((group["status"] == "NOT_RUN").sum()),
        "balanced_accuracy": bacc,
    })

metrics = pd.DataFrame(metric_rows)
macro_bacc = float(metrics["balanced_accuracy"].mean()) if metrics["balanced_accuracy"].notna().any() else None
metrics["macro_balanced_accuracy"] = macro_bacc
atomic_save(metrics, METRICS_PATH)
atomic_save(merged, FINAL_PATH)
display(metrics)
print("Macro Balanced Accuracy:", macro_bacc)

try:
    import vllm
    vllm_version = vllm.__version__
except Exception:
    vllm_version = None

gpu_snapshot = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=True).stdout
(OUTPUT_DIR / "nvidia_smi_final.txt").write_text(gpu_snapshot, encoding="utf-8")
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "HalluGuard through MiniCheck, deterministic 100-per-dataset sample",
    "minicheck_commit": MINICHECK_COMMIT,
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset": DATASET_ID,
    "split": SPLIT,
    "dataset_rows_total": len(full_df),
    "datasets": dataset_names,
    "sample_per_dataset": SAMPLE_PER_DATASET,
    "sample_rows": len(eval_df),
    "source_index_preserved": True,
    "random_state": RANDOM_STATE,
    "chunk_size": CHUNK_SIZE,
    "max_model_len": MAX_MODEL_LEN,
    "max_tokens": MAX_TOKENS,
    "generation": {"enable_thinking": True, "temperature": 0.6, "top_p": 0.95, "top_k": 20, "seed": 42},
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "enable_prefix_caching": ENABLE_PREFIX_CACHING,
    "vllm_engine": {
        "version": "0.10.2",
        "enforce_eager": True,
        "disable_custom_all_reduce": True,
        "gpu_memory_utilization": 0.80,
        "max_num_seqs": 16,
        "reason": "Kaggle dual-T4 CUDA-graph/custom-all-reduce stability",
    },
    "threshold": "> 0.5 (MiniCheck unchanged)",
    "metric": "Balanced Accuracy; unweighted macro across datasets",
    "sub_batch_size": SUB_BATCH_SIZE,
    "valid_predictions": int((merged["status"] == "OK").sum()),
    "parse_failures": int((merged["status"] == "PARSE_FAILURE").sum()),
    "inference_failures": int((merged["status"] == "INFERENCE_FAILURE").sum()),
    "not_run": int((merged["status"] == "NOT_RUN").sum()),
    "macro_balanced_accuracy": macro_bacc,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "vllm": vllm_version,
    "cuda": torch.version.cuda,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "artifacts": {
        "checkpoint": str(CHECKPOINT_PATH),
        "final_predictions": str(FINAL_PATH),
        "dataset_metrics": str(METRICS_PATH),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

if manifest["not_run"]:
    print("The run is incomplete. Re-run this notebook to resume from the checkpoint.")
else:
    print("All sampled rows were attempted. Failures, if any, remain explicit and have no prediction.")
